# Bangladesh Highway Vehicle Detection — Kaggle runner

End-to-end: **prepare → EDA → train (YOLO11-m @1280) → validate+tune → infer → submission**.

### Setup (once)
1. Right panel → **Add Input** → add the competition data (the folder containing `train/` and `test/`).
2. Settings → **Accelerator: GPU** (T4 ×2 or P100) and **Internet: ON** (to fetch the repo + pretrained weights).
3. The data path is **auto-detected** — no manual editing.

> ⚠️ **The code lives in GitHub, not in this notebook.** Cell 1 does `git clone` of `mewhatever/DUET_DATATHON` to get `src/` + `configs/`. If you change any `src/*.py` locally you **must push to GitHub first**, otherwise Kaggle clones the old code. (Repo already on disk? delete `/kaggle/working/DUET_DATATHON` or `git pull` to refresh.)

### Which cells to run — pick ONE path

**Setup (always):** run **cell 0, 1, 2**, then **3a** (smoke, ~2 min — confirms the chain works before you burn hours).

| Path | Run | Skip | Time | Use when |
|---|---|---|---|---|
| **A · Baseline** (single model) | 3b → 4 → 5 → 6 | the 5-fold cell | ~2.5 h | first submission / quick score on the board |
| **B · Ensemble** (best score) | the **5-fold cell** (last cell) | 3b, 4, 5, 6 | ~7 h | final push — 5-fold + WBF, supersedes the baseline |

Both write `/kaggle/working/submission.csv` with grader-ready formatting (bare-stem `image_id`, confidence in `[0,1]`). **Submit that file.** Don't run both A and B in one session — B retrains fold 0 anyway, so it would waste the baseline's time.

In [ ]:
# --- 0. Extras Kaggle may lack (torch / ultralytics are usually preinstalled) ---
!pip -q install ensemble-boxes >/dev/null 2>&1
!pip -q install -U ultralytics albumentations >/dev/null 2>&1
import torch; print('CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())

In [ ]:
# --- 1. Repo + AUTO-DETECT the competition data --------------------------------
# Root cause of the earlier failure: glob('/kaggle/input/*')[0] grabbed Kaggle's
# internal 'competitions' folder instead of the data. Robust fix: find train.csv
# wherever Kaggle mounted it, then build a NORMALIZED canonical tree at
# /kaggle/working/comp  (train/{train.csv,images}, test/{images,sample_submission.csv})
# via symlinks, so every downstream step sees the exact layout it expects.
import os, shutil
from pathlib import Path

PROJECT_DIR = '/kaggle/working/DUET_DATATHON'        # repo: src/, configs/
if not os.path.isdir(PROJECT_DIR):
    !git clone -q https://github.com/mewhatever/DUET_DATATHON.git {PROJECT_DIR}
%cd $PROJECT_DIR

INPUT_ROOT = '/kaggle/input'

def _find_files(root, name):
    hits = []
    for dp, _dns, fns in os.walk(root):
        if name in fns:
            hits.append(Path(dp) / name)
    return sorted(hits, key=lambda p: len(str(p)))      # shallowest first

def _jpg_dirs(root, min_count=5):
    out = {}
    for dp, _dns, fns in os.walk(root):
        n = sum(1 for f in fns if f.lower().endswith('.jpg'))
        if n >= min_count:
            out[Path(dp)] = n
    return out

# locate the assets anywhere under /kaggle/input
csvs    = _find_files(INPUT_ROOT, 'train.csv')
samples = _find_files(INPUT_ROOT, 'sample_submission.csv')
assert csvs, ("train.csv not found under /kaggle/input — add the competition data "
              "via the right-hand 'Add Input' panel, then re-run.")
train_csv  = csvs[0]
sample_csv = samples[0] if samples else None

jpg_dirs = _jpg_dirs(INPUT_ROOT)
print('jpg dirs found under /kaggle/input:')
for d, n in sorted(jpg_dirs.items(), key=lambda kv: -kv[1]):
    print(f'   {n:6d}  {d}')

train_dirs = {d: n for d, n in jpg_dirs.items() if 'train' in str(d).lower() and 'test' not in str(d).lower()}
test_dirs  = {d: n for d, n in jpg_dirs.items() if 'test'  in str(d).lower()}
# fallback: no train/test keywords but exactly two image dirs -> larger=train, smaller=test
if not (train_dirs and test_dirs) and len(jpg_dirs) == 2:
    big, small = sorted(jpg_dirs, key=lambda d: -jpg_dirs[d])
    train_dirs, test_dirs = {big: jpg_dirs[big]}, {small: jpg_dirs[small]}
    print('NOTE: no train/test keyword in paths; assigned by size (larger=train).')
assert train_dirs, f"could not identify the TRAIN image dir among {list(jpg_dirs)}"
assert test_dirs,  f"could not identify the TEST image dir among {list(jpg_dirs)}"
train_img = max(train_dirs, key=train_dirs.get)
test_img  = max(test_dirs,  key=test_dirs.get)

# build the normalized canonical tree (idempotent: safe to re-run)
COMP_DIR = '/kaggle/working/comp'
def _relink(target, link):
    link = Path(link); link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        link.unlink()
    elif link.is_dir():
        shutil.rmtree(link)
    elif link.exists():
        link.unlink()
    os.symlink(Path(target).resolve(), link)

Path(COMP_DIR, 'train').mkdir(parents=True, exist_ok=True)
Path(COMP_DIR, 'test').mkdir(parents=True, exist_ok=True)
_relink(train_csv, f'{COMP_DIR}/train/train.csv')
_relink(train_img, f'{COMP_DIR}/train/images')
_relink(test_img,  f'{COMP_DIR}/test/images')
if sample_csv:
    _relink(sample_csv, f'{COMP_DIR}/test/sample_submission.csv')

n_train = len(list(Path(COMP_DIR, 'train', 'images').glob('*.jpg')))
n_test  = len(list(Path(COMP_DIR, 'test',  'images').glob('*.jpg')))
print('\nPROJECT_DIR =', PROJECT_DIR)
print('COMP_DIR    =', COMP_DIR, '  (normalized symlinks ->)')
print('  train.csv ->', train_csv)
print(f'  train/img -> {train_img}  ({n_train} jpg)')
print(f'  test/img  -> {test_img}  ({n_test} jpg)')
print('  sample    ->', sample_csv)
assert n_train > 0 and n_test > 0, "no images linked — inspect the jpg-dir list above"
print('\nOK: expecting ~810 train / ~327 test images for this competition.')

In [ ]:
# --- 2. Build YOLO labels + frame-phase folds + configs/data.yaml --------------
# Regenerates configs/data.yaml with correct Kaggle paths (was the stale Mac paths).
!python -m src.prepare_data --data-root "$COMP_DIR" --out /kaggle/working/yolo --folds 5 --val-fold 0
!python -m src.eda          --data-root "$COMP_DIR" --out outputs/eda
import yaml; print('\nconfigs/data.yaml ->'); print(open('configs/data.yaml').read())

In [ ]:
# --- 3a. SMOKE TEST (run this first) -------------------------------------------
# Fast 1-epoch run on ONE GPU to confirm training works end-to-end (~2 min).
# Tiny imgsz/batch on purpose. Writes a SEPARATE checkpoint (name=smoke), so it
# never collides with the real run below.
!python -m src.train --config configs/train_baseline.yaml --data configs/data.yaml \
    --name smoke --device 0 --imgsz 640 --epochs 1 --batch 4
import os
print('smoke best.pt exists:', os.path.exists('checkpoints/smoke/weights/best.pt'))

In [ ]:
# --- 3b. FULL RUN — both T4 GPUs (~2x faster) ----------------------------------
# YOLO11-m @1280, 120 epochs. --device 0,1 uses BOTH GPUs; batch 8 = 4 per GPU.
# NOTE: a Kaggle T4 has ~14.5GB, so 8 imgs/GPU at 1280 OOMs -> keep 4/GPU (batch 8).
# Do NOT lower imgsz below 1280 (tiny vehicles need it). Run AFTER smoke passes.
# If you hit a *system* RAM OOM, set `cache: false` in configs/train_baseline.yaml.
!python -m src.train --config configs/train_baseline.yaml --data configs/data.yaml \
    --name e00_yolo11m_1280 --device 0,1 --batch 8
BEST = 'checkpoints/e00_yolo11m_1280/weights/best.pt'
import os; assert os.path.exists(BEST), f'training did not produce {BEST}'
print('best.pt ->', BEST)

In [ ]:
# --- 4. Validate + tune NMS IoU -> configs/inference.yaml (NON-FATAL) ----------
# If tuning errors (e.g. transient GPU hiccup), we keep the existing inference.yaml
# defaults so steps 5-6 can still produce a submission.
!python -m src.validate --weights $BEST --data configs/data.yaml --tune --save configs/inference.yaml \
    || echo "[warn] tuning failed -> using the existing configs/inference.yaml defaults"
import os; assert os.path.exists('configs/inference.yaml'), 'configs/inference.yaml is missing'

In [ ]:
# --- 5. Inference on the 327 test images (TTA) -> prediction cache -------------
!python -m src.inference --weights $BEST --data-root "$COMP_DIR" \
    --infer-config configs/inference.yaml --out outputs/predictions/e00.pkl --tta
import os; assert os.path.exists('outputs/predictions/e00.pkl'), \
    'inference cache was NOT created — check the log above for the real error'

In [ ]:
# --- 6. Build the submission CSV (exact competition format; all 327 rows) ------
# sample_submission.csv is only a truncated 10-row example, so the COMPLETE id
# list comes from --test-images (the 327 test jpgs).
# Grader format is baked into the code now: image_id = BARE STEM (no .jpg/.txt) and
# confidence in [0,1] (src/submission.py + configs/inference.yaml). The asserts below
# re-verify it so a malformed CSV can never be submitted by accident.
!python -m src.submission --preds outputs/predictions/e00.pkl \
    --infer-config configs/inference.yaml \
    --test-images "$COMP_DIR/test/images" \
    --sample "$COMP_DIR/test/sample_submission.csv" \
    --out /kaggle/working/submission.csv
import os, pandas as pd
assert os.path.exists('/kaggle/working/submission.csv'), 'submission.csv was NOT created'
sub = pd.read_csv('/kaggle/working/submission.csv')
print('submission shape:', sub.shape, '(expect 327 rows)')
assert len(sub) == 327, f'expected 327 rows, got {len(sub)}'
# grader matches on the bare stem -> id must NOT end in a file extension
bad = sub['image_id'].astype(str).str.contains(r'\.(jpg|jpeg|png|txt)$', case=False, regex=True).sum()
assert bad == 0, f'{bad} image_ids still carry a file extension — grader would reject'
print('id[0]:', sub['image_id'].iloc[0], '(no extension ✓)')
print(sub.head().to_string())
!ls -la /kaggle/working/submission.csv

## Path B — 5-fold + WBF ensemble (best score, one cell)

Trains all 5 phase-folds, predicts with each, fuses with **Weighted Box Fusion**, and writes the final submission — all in the single cell below.

**Prereq:** run cells **0, 1, 2** (and **3a** smoke) once first. Then run this cell on its own — **skip cells 3b/4/5/6**, this replaces them. Each fold trains in its own process (multi-GPU DDP-safe), and re-running **skips folds already trained**, so you can resume after a Kaggle timeout. ~7 h at `EPOCHS=80` (fits the 12 h limit).

In [ ]:
# ============================================================================
# PATH B — SCALE-UP: 5-fold train + per-fold inference + WBF + submission.
# Run cells 0,1,2,3a first, then THIS cell alone. Skip 3b/4/5/6 (this supersedes
# them). Each fold trains in its OWN process (DDP-safe). Re-running SKIPS folds
# already trained, so if you hit Kaggle's 12h wall just re-run in a new session
# and it resumes. ~7h total at EPOCHS=80 on T4x2 (fits 12h with margin).
# ============================================================================
import os, glob
assert 'COMP_DIR' in globals() and 'PROJECT_DIR' in globals(), \
    "Run cells 0-2 first (they define COMP_DIR / PROJECT_DIR and prepare the data)."
os.chdir(PROJECT_DIR)

EXP     = 'e07_yolo11m'
N_FOLDS = 5
DEVICE  = '0,1'      # both T4s; use '0' for a single GPU / P100
EPOCHS  = 80         # 5x80ep ~7h (fits 12h). 120 ~10-11h for <1 mAP more -> not worth the wall-clock risk.

# fold splits live in /kaggle/working/yolo (cell 2). Rebuild if missing.
if len(glob.glob('/kaggle/working/yolo/train_fold*.txt')) < N_FOLDS:
    !python -m src.prepare_data --data-root "$COMP_DIR" --out /kaggle/working/yolo --folds {N_FOLDS} --val-fold 0

# 1) TRAIN each fold in its own process (skip if already trained) --------------
for k in range(N_FOLDS):
    best_k = f'checkpoints/{EXP}_fold{k}/weights/best.pt'
    if os.path.exists(best_k):
        print(f'[fold {k}] already trained -> {best_k} (skip)'); continue
    print(f'\n===== TRAIN fold {k} / {N_FOLDS-1} =====')
    !python -m src.train --config configs/train_baseline.yaml --data configs/data.yaml --name {EXP}_fold{k} --folds 1 --device {DEVICE} --epochs {EPOCHS}
    assert os.path.exists(best_k), f'fold {k} did not produce {best_k}'

# 2) PER-FOLD inference on the 327 test images (TTA) ---------------------------
for k in range(N_FOLDS):
    w = f'checkpoints/{EXP}_fold{k}/weights/best.pt'
    !python -m src.inference --weights {w} --data-root "$COMP_DIR" --infer-config configs/inference.yaml --out outputs/predictions/fold{k}.pkl --tta
    assert os.path.exists(f'outputs/predictions/fold{k}.pkl'), f'fold {k} inference cache missing'

# 3) WBF-fuse the 5 caches -----------------------------------------------------
caches = ' '.join(f'outputs/predictions/fold{k}.pkl' for k in range(N_FOLDS))
!python -m src.ensemble --preds {caches} --out outputs/predictions/wbf.pkl --iou 0.55 --skip 0.001

# 4) SUBMISSION (bare-stem ids + [0,1] conf are baked into the code) -----------
!python -m src.submission --preds outputs/predictions/wbf.pkl \
    --infer-config configs/inference.yaml \
    --test-images "$COMP_DIR/test/images" \
    --sample "$COMP_DIR/test/sample_submission.csv" \
    --out /kaggle/working/submission.csv

import pandas as pd
sub = pd.read_csv('/kaggle/working/submission.csv')
print('\nsubmission shape:', sub.shape, '(expect 327 rows)')
assert len(sub) == 327, f'expected 327 rows, got {len(sub)}'
bad = sub['image_id'].astype(str).str.contains(r'\.(jpg|jpeg|png|txt)$', case=False, regex=True).sum()
assert bad == 0, f'{bad} image_ids still carry a file extension'
print('id[0]:', sub['image_id'].iloc[0], '(no extension ✓)')
print(sub.head().to_string())